In [0]:
%run ../config/config

In [0]:
import sys
from pathlib import Path

SRC_PATH = str(Path.cwd().parent / "src")
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

from gold import build_gold_tables
from pyspark.sql import functions as F

spark.conf.set("spark.databricks.execution.timeout", "18000")

In [0]:
SILVER_TABLE_PATH = f"{CATALOG}.{SILVER_SCHEMA}.{SILVER_TABLE}"
GOLD_TABLE_PREFIX = f"{CATALOG}.{GOLD_SCHEMA}"

df_silver = spark.table(SILVER_TABLE_PATH)

gold_tables = build_gold_tables(
    silver_df=df_silver,
    fact_product_entries_cols=FACT_PRODUCT_ENTRIES_COLS,
    dim_brands_cols=DIM_BRANDS_COLS,
    dim_products_cols=DIM_PRODUCTS_COLS,
)

table_names = {
    "fact_product_entries": GOLD_FACT_PRODUCT_ENTRIES,
    "dim_brands": GOLD_DIM_BRANDS,
    "dim_products": GOLD_DIM_PRODUCTS,
}

for key, df in gold_tables.items():
    (
        df.write
        .mode("overwrite")
        .format("delta")
        .option("mergeSchema", "true")
        .saveAsTable(f"{GOLD_TABLE_PREFIX}.{table_names[key]}")
    )